# 03 · Phasor analysis

Donor phasors behind the curated masks, calibrated and reduced to phase
lifetime and apparent FRET efficiency.

1. **Extraction** – every `.ptu` is reconstructed behind its IB and its
   cytosol mask (donor channel, z planes summed).
2. **Session correction** – one phasor rotation per imaging session, from the
   rising edge of the accumulated IB decays.
3. **Calibration** – one absolute factor from the donor-only sample.
4. **Quantification** – $\tau_\varphi$, $\tau_m$, $E_\varphi$ and $E_m$ per
   object, image and construct.

Written to `results/`: `phasors.csv`, `decays.csv`, `fret_objects.csv`,
`fret_images.csv`, `fret_constructs.csv` and the diagnostic figures.
Notebooks 04 and 05 read these tables.

In [ ]:
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import io

from ptuio.reader import TTTRReader
from ptuio.reconstructor import ImageReconstructor

import config
from helpers import (
    apparent_lifetimes,
    calculate_phasor_from_lifetime,
    draw_universal_circle,
    fit_leading_edge,
    index_masks,
    index_samples,
    leading_edge_crossing,
    load_sample_rename,
    make_scan_config,
    read_acquisition_params,
    rise_baseline,
    run_reconstruction,
    save_figure,
)

plt.rcParams['svg.fonttype'] = 'none'   # keep SVG text editable

## Inputs

A **sample** is one folder of acquisitions (one construct); a **session** is
the imaging day it was recorded on. All samples of a session share one
instrument state and therefore one correction factor.

In [ ]:
# ----------------------------------------------------------------- data in --
raw_dir           = config.RAW_DIR            # <session>/<sample>/*.ptu
mask_dir          = config.MASKS_DIR          # curated masks
sample_rename_csv = config.SAMPLE_NAMES_CSV   # folder name -> construct name
output_dir        = config.RESULTS_DIR

# ------------------------------------------------------------- acquisition --
target_bin_width_ps = 40   # TCSPC bin width the accumulated decays are binned to
frames              = 5    # z planes per acquisition
donor_channel       = 1    # detector channel of the donor (mTurquoise2)

# -------------------------------------------------------------- calibration --
# Donor-only sample: no acceptor, so its phasor has to sit on the universal
# circle at the unquenched donor lifetime (Goedhart et al., Nat Commun 2012).
donor_sample      = 'Turq'
donor_mask_kind   = 'cell'
donor_lifetime_ns = 3.8

# ------------------------------------------------------- session correction --
edge_mask_kind   = 'ib'    # masks whose accumulated decay gives the timing
edge_fractions   = (0.05, 0.10, 0.15, 0.20, 0.25)   # of the peak height
edge_min_photons = 20_000  # decays below this are left out of the session median

# ----------------------------------------------------------- quantification --
min_photons_per_object = 300   # dimmer objects are dropped at every level

# ------------------------------------------------------------------- plots --
zoom_padding = 0.1   # margin of the cropped phasor plot, as a fraction of the data span

figure_dir = os.path.join(output_dir, "figures")
os.makedirs(figure_dir, exist_ok=True)

## Samples, masks and display names

Folder names are the internal key. Display names from `rename_key.csv` are
applied only to what is written out; the tables keep the folder name in
`sample_folder`.

In [ ]:
MASK_LABEL = {'ib': 'IB', 'cell': 'cytosol'}

samples       = index_samples(raw_dir)
masks         = index_masks(mask_dir, samples)
sample_date   = {sample: info['session'] for sample, info in samples.items()}
sample_rename = load_sample_rename(sample_rename_csv)


def display_name(sample):
    """The name `sample` is exported and plotted under."""
    return sample_rename.get(sample, sample)


def display_order(sample):
    """Natural sort key on the display name: H1-2 before H1-10."""
    return tuple(int(part) if part.isdigit() else part.lower()
                 for part in re.split(r'(\d+)', display_name(sample)))


def with_display_names(df, column='sample'):
    """Copy of `df` for export: display names in `column`, folder name alongside."""
    if column not in df.columns or df.empty:
        return df
    out = df.copy()
    out.insert(out.columns.get_loc(column) + 1, 'sample_folder', out[column])
    out[column] = out[column].map(display_name)
    return out


unmasked = sorted(s for s in samples if not any(key[0] == s for key in masks))
print(f"{len(samples)} sample(s) in {len(set(sample_date.values()))} session(s), "
      f"{len(masks)} mask file(s)")
if unmasked:
    print(f"No curated masks, skipped: {unmasked}")

## Acquisition parameters

Read from the header of the first `.ptu` of each sample. Sync rate and TCSPC
resolution differ between sessions, so nothing is hard-coded.

In [ ]:
sample_params = {}
for sample, info in samples.items():
    per_file = [read_acquisition_params(TTTRReader(os.path.join(info['path'], f)).header.tags,
                                        target_bin_width_ps)
                for f in info['files']]
    if len({p['tcspc_resolution'] for p in per_file}) > 1 and sample not in unmasked:
        raise ValueError(f"sample '{sample}' mixes TCSPC resolutions")
    sample_params[sample] = per_file[0]

for sample in unmasked:
    del samples[sample], sample_params[sample], sample_date[sample]

summary = pd.DataFrame([{
    'session':       sample_date[s],
    'sample':        s,
    'files':         len(samples[s]['files']),
    'sync_rate_Hz':  p['sync_rate'],
    'resolution_ps': p['tcspc_resolution'] * 1e12,
    'binning':       p['tcspc_binning'],
    'bins':          p['n_bins'],
} for s, p in sample_params.items()])
display(summary)

# Used only to draw the universal circle.
reference_sync_rate = next(iter(sample_params.values()))['sync_rate']

## Extract decays and per-object phasors

Each file is reconstructed once per mask. A pass returns the accumulated
decay of the masked region and the per-pixel photon counts and phasor sums.

The phasor of any group of pixels is the sum of their phasor sums over the
sum of their counts. Objects, images and constructs are therefore all pooled
from the same per-pixel arrays, and the result equals the phasor of the
summed decay. Phasor sums are accumulated at the raw TCSPC resolution; the
binned decay is used only for the rising-edge timing and the exported curves.

In [ ]:
def extract_masked(ptu_path, mask, params):
    """Binned decay, per-pixel photon counts and per-pixel phasor sums (donor channel)."""
    reader = TTTRReader(ptu_path)
    reconstructor = ImageReconstructor(
        config=make_scan_config(reader.header.tags, *mask.shape, frames=frames),
        omega=2 * np.pi * params['sync_rate'] * params['tcspc_resolution'],
        tcspc_channels=params['tcspc_bins'],
        roi_mask=mask > 0,
        outputs=['tcspc_histogram', 'photon_count', 'phasor'],
    )

    result = run_reconstruction(reader, reconstructor, params['wrap'])

    if result.photon_count.sizes['channel'] <= donor_channel:   # no donor photons at all
        return (np.zeros(params['n_bins'], dtype=np.int64),
                np.zeros(mask.shape), np.zeros(mask.shape, dtype=complex))

    decay = result.tcspc_histogram.coarsen(frame=frames).sum().isel(frame=0, channel=donor_channel)
    # 'trim' drops trailing channels that do not fill a whole bin.
    decay = decay.coarsen(tcspc_channel=params['tcspc_binning'], boundary='trim').sum()
    decay = np.asarray(decay, dtype=np.int64)

    counts = np.asarray(result.photon_count.isel(sequence=0, channel=donor_channel),
                        dtype=np.float64)
    g = np.asarray(result.phasor_g.isel(sequence=0, channel=donor_channel), dtype=np.float64)
    s = np.asarray(result.phasor_s.isel(sequence=0, channel=donor_channel), dtype=np.float64)

    # phasor_g/s are per-pixel means (NaN where empty); times the count gives sums.
    phasor_sum = np.nan_to_num(g + 1j * s) * counts

    # The mask is 2D and applies to every z plane, so the planes are summed.
    return decay, counts.sum(axis=0), phasor_sum.sum(axis=0)


def pooled_phasor(rows):
    """Photon-weighted phasor of a group of records, and its photon count."""
    photons = sum(r['photons'] for r in rows)
    if not photons:
        return np.nan + 1j * np.nan, 0
    return sum(r['phasor_sum'] for r in rows) / photons, photons


def aggregate_decay(records, sample, mask_type):
    """Sum of all mask decays of one sample and mask type (None if there are none)."""
    selected = [r['decay'] for r in records
                if r['sample'] == sample and r['mask_type'] == mask_type]
    return np.sum(selected, axis=0) if selected else None


records, objects = [], []   # one row per mask file / per labelled object

for sample, info in samples.items():
    params = sample_params[sample]

    for ptu_file in info['files']:
        base = os.path.splitext(ptu_file)[0]

        for kind, mask_type in MASK_LABEL.items():
            mask_path = masks.get((sample, base, kind))
            if mask_path is None:
                continue

            mask = io.imread(mask_path)
            decay, counts, phasor_sum = extract_masked(
                os.path.join(info['path'], ptu_file), mask, params)

            common = {
                'date':      sample_date[sample],
                'sample':    sample,
                'mask_type': mask_type,
                'ptu_file':  ptu_file,
                'mask_file': os.path.basename(mask_path),
            }

            # Curated masks can have gaps in their label numbering.
            labels = np.unique(mask[mask > 0])
            for label in labels:
                selected = mask == label
                objects.append({
                    **common,
                    'label':      int(label),
                    'pixels':     int(selected.sum()),
                    'photons':    int(counts[selected].sum()),
                    'phasor_sum': phasor_sum[selected].sum(),
                })

            records.append({
                **common,
                'objects':    len(labels),
                'photons':    int(counts.sum()),
                'phasor_sum': phasor_sum.sum(),
                'decay':      decay,
            })

for row in records + objects:
    row['phasor_raw'], _ = pooled_phasor([row])

print(f"Extracted {len(records)} mask decays and {len(objects)} objects "
      f"from {len(samples)} sample(s).")

## Session correction: timing offset from the rising edge

A decay delayed by $\Delta t$ has its phasor multiplied by
$e^{+i\omega\Delta t}$: a rotation, with no change in modulation. The delay
of each session is read from the rising edge of the accumulated decays
(baseline-subtracted, normalised to the peak, mean of the interpolated
crossings at `edge_fractions`). This part of the rise is set by the
instrument response and is nearly independent of the lifetime. The session
value is the median over its samples, and the correction factor is

$$c^\text{corr}_\text{session} = e^{-i\,\omega\,t_0(\text{session})}$$

Only ratios between sessions are used below, so the arbitrary zero of $t_0$
cancels.

In [ ]:
def sample_edge_times(mask_kind):
    """Rising-edge timing of every sample's accumulated decay of one mask type.

    Also returns the cross-checks used in the diagnostics: the timing and IRF
    width from an EMG fit, the uniform background fraction and the sample's
    own phase.
    """
    label = MASK_LABEL[mask_kind]
    rows = []

    for sample in sorted({r['sample'] for r in records if r['mask_type'] == label}):
        decay = aggregate_decay(records, sample, label)
        if decay is None or decay.sum() < edge_min_photons:
            continue

        params = sample_params[sample]
        t0, spread = leading_edge_crossing(decay, params['bin_width'],
                                           fractions=edge_fractions, return_spread=True)
        if not np.isfinite(t0):
            print(f"WARNING: no usable rise for '{sample}' ({label}) -- skipped")
            continue

        try:
            fitted = fit_leading_edge(decay, params['bin_width'])
            t0_fit, sigma = fitted['t0'], fitted['sigma']
        except Exception as error:
            print(f"NOTE: EMG cross-check failed for '{sample}' ({label}): {error}")
            t0_fit, sigma = np.nan, np.nan

        baseline, _, _ = rise_baseline(decay)
        phasor, _ = pooled_phasor(
            [r for r in records if r['sample'] == sample and r['mask_type'] == label])

        rows.append({
            'date':         sample_date[sample],
            'sample':       sample,
            'mask_type':    label,
            'photons':      int(decay.sum()),
            't0_ps':        t0 * 1e12,
            'spread_ps':    spread * 1e12,
            't0_fit_ps':    t0_fit * 1e12,
            'sigma_ps':     sigma * 1e12,
            'f_background': baseline * len(decay) / decay.sum(),
            'phase_deg':    np.degrees(np.angle(phasor)),
            'sync_rate':    params['sync_rate'],
        })

    return pd.DataFrame(rows)


def robust_sd(values):
    """MAD-based standard deviation."""
    values = np.asarray(values, dtype=float)
    return 1.4826 * np.median(np.abs(values - np.median(values)))


edge_samples = pd.concat([sample_edge_times(kind) for kind in MASK_LABEL], ignore_index=True)
primary = edge_samples[edge_samples['mask_type'] == MASK_LABEL[edge_mask_kind]]

correction_factor = {}
day_rows = []
for date, group in primary.groupby('date'):
    t0_ps = float(np.median(group['t0_ps']))
    sd    = robust_sd(group['t0_ps'])
    omega = 2 * np.pi * float(np.median(group['sync_rate']))

    # A later pulse means a larger phase, so the correction rotates clockwise.
    correction_factor[date] = np.exp(-1j * omega * t0_ps * 1e-12)
    day_rows.append({
        'date':         date,
        'n_samples':    len(group),
        't0_ps':        t0_ps,
        'sd_ps':        sd,
        'sem_ps':       sd / np.sqrt(len(group)),
        'sigma_ps':     float(np.median(group['sigma_ps'])),
        'f_background': float(np.median(group['f_background'])),
    })

edge_days = pd.DataFrame(day_rows).sort_values('date').reset_index(drop=True)

# Shifts are quoted against the session of the donor-only sample.
anchor_date  = sample_date[donor_sample]
reference_t0 = float(edge_days.loc[edge_days['date'] == anchor_date, 't0_ps'].iloc[0])
edge_days['shift_ps']     = edge_days['t0_ps'] - reference_t0
edge_days['rotation_deg'] = -np.degrees(
    2 * np.pi * reference_sync_rate * edge_days['shift_ps'] * 1e-12)

print(f"Timing offsets from the {MASK_LABEL[edge_mask_kind]} rising edge, "
      f"relative to {anchor_date}:")
edge_days[['date', 'n_samples', 't0_ps', 'sd_ps', 'sem_ps', 'shift_ps', 'rotation_deg']].round(1)

### Diagnostics

1. **Estimators agree** – the crossings and an independent EMG fit of the
   rise give the same session shifts.
2. **Photon populations agree** – IB and cytosol decays see the same
   instrument and give the same shifts.
3. **$t_0$ does not follow the lifetime** – within a session, the slope of
   $t_0$ against the sample's own phase is close to zero.
4. **Modulation is stable** – the IRF width and the uniform background, the
   two quantities that set the modulation, barely change between sessions,
   so a rotation alone is sufficient.

The plot shows the accumulated rises as measured and after alignment on $t_0$.

In [ ]:
omega_reference = 2 * np.pi * reference_sync_rate


def session_shifts(frame, column):
    """Session median of `column`, relative to the anchor session."""
    median = frame.groupby('date')[column].median()
    return median - median[anchor_date]


# --- 1 + 2: estimators and photon populations --------------------------------
checks = pd.DataFrame({
    'crossings (primary)': session_shifts(primary, 't0_ps'),
    'EMG fit':             session_shifts(primary, 't0_fit_ps'),
})
for label in sorted(edge_samples['mask_type'].unique()):
    checks[f'crossings, {label}'] = session_shifts(
        edge_samples[edge_samples['mask_type'] == label], 't0_ps')

print(f"Session shift relative to {anchor_date} (ps):\n")
print(checks.round(1).to_string())
disagreement = float(np.nanmax(checks.max(axis=1) - checks.min(axis=1)))
print(f"\nlargest disagreement between columns: {disagreement:.1f} ps "
      f"({np.degrees(omega_reference * disagreement * 1e-12):.2f} deg)")

# --- 3: sensitivity of t0 to the sample's own phase --------------------------
print("\nSlope of t0 against the sample's own phase, within a session:\n")
for (date, label), group in edge_samples.groupby(['date', 'mask_type']):
    if len(group) < 4:
        continue
    slope, intercept = np.polyfit(group['phase_deg'], group['t0_ps'], 1)
    residual = np.std(group['t0_ps'] - (slope * group['phase_deg'] + intercept))
    print(f"  {date}  {label:8s} n={len(group):2d}  "
          f"phase range {np.ptp(group['phase_deg']):5.1f} deg  "
          f"slope {slope:+6.2f} ps/deg  residual {residual:5.1f} ps")

# --- 4: what sets the modulation ---------------------------------------------
audit = edge_days[['date', 'sigma_ps', 'f_background']].copy()
audit['from_irf_width']  = np.exp(-0.5 * (omega_reference * audit['sigma_ps'] * 1e-12) ** 2)
audit['from_background'] = 1.0 - audit['f_background']
audit['combined']        = audit['from_irf_width'] * audit['from_background']
print("\nModulation factors that the rotation does not correct:\n")
print(audit.round(4).to_string(index=False))
print(f"\nspread of the combined factor across sessions: {np.ptp(audit['combined']):.4f}")

# --- the rises, as measured and aligned --------------------------------------
label   = MASK_LABEL[edge_mask_kind]
colours = {date: plt.get_cmap('tab10')(i) for i, date in enumerate(edge_days['date'])}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for date in edge_days['date']:
    members = primary.loc[primary['date'] == date, 'sample']
    # Decays binned from 5 ps and recorded at 40 ps differ by one channel in length.
    decays   = [aggregate_decay(records, s, label) for s in members]
    shortest = min(len(x) for x in decays)
    decay    = np.sum([x[:shortest] for x in decays], axis=0)

    baseline, peak_index, height = rise_baseline(decay)
    time_ps = np.arange(len(decay)) * sample_params[members.iloc[0]]['bin_width'] * 1e12
    rise    = (decay - baseline) / height
    window  = slice(max(0, peak_index - 40), peak_index + 15)
    shift   = float(edge_days.loc[edge_days['date'] == date, 't0_ps'].iloc[0])

    axes[0].plot(time_ps[window], rise[window], color=colours[date], label=date)
    axes[1].plot(time_ps[window] - shift, rise[window], color=colours[date], label=date)

for ax, title in zip(axes, (f'{label} rise, as measured', f'{label} rise, aligned on $t_0$')):
    ax.axhspan(min(edge_fractions), max(edge_fractions), color='grey', alpha=0.12,
               label='crossing band' if ax is axes[0] else None)
    ax.set_xlabel('time (ps)')
    ax.set_ylabel('fraction of peak')
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(title)
    ax.legend(fontsize=8)

fig.tight_layout()
save_figure(fig, "stage1_leading_edge_rises", figure_dir)
plt.show()

## Calibration: the donor-only sample

All donor masks are pooled into one phasor, and

$$c^\text{cal} = \frac{\Phi_\text{theory}(\tau_D)}{\Phi_\text{measured}}
    \qquad\text{with}\qquad
    \Phi_\text{theory}(\tau) = \frac{1}{1 - i\,\omega\tau}$$

places the unquenched donor on the universal circle at $\tau_D$. The donor is
assumed to be mono-exponential, and absolute lifetimes are only as accurate
as `donor_lifetime_ns`; differences between samples do not depend on it.

In [ ]:
donor_records = [r for r in records
                 if r['sample'] == donor_sample and r['mask_type'] == MASK_LABEL[donor_mask_kind]]
donor_params  = sample_params[donor_sample]

phasor_measured, donor_photons = pooled_phasor(donor_records)
phasor_theory = calculate_phasor_from_lifetime(donor_lifetime_ns * 1e-9,
                                               donor_params['sync_rate'])
calibration_factor = phasor_theory / phasor_measured

print(f"Donor '{donor_sample}' ({anchor_date}): {len(donor_records)} mask(s), "
      f"{donor_photons:,} photons")
print(f"measured phasor:  g={phasor_measured.real:.4f}  s={phasor_measured.imag:.4f}")
print(f"target phasor:    g={phasor_theory.real:.4f}  s={phasor_theory.imag:.4f}   "
      f"(tau = {donor_lifetime_ns:g} ns)")
print(f"calibration factor: rotation {np.angle(calibration_factor, deg=True):+.2f} deg, "
      f"modulation x{abs(calibration_factor):.4f}")

donor_decay = np.sum([r['decay'] for r in donor_records], axis=0)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].semilogy(np.arange(len(donor_decay)) * donor_params['bin_width'] * 1e9,
                 donor_decay, color='tab:blue')
axes[0].set_xlabel("time (ns)")
axes[0].set_ylabel("counts")
axes[0].set_title(f"{anchor_date} -- accumulated donor-only decay")

draw_universal_circle(axes[1], sync_rate=donor_params['sync_rate'], tau_max=5,
                      xlim=(0, 1), ylim=(0, 0.7))
axes[1].set_aspect('equal', adjustable='box')
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 0.7)
axes[1].annotate("", xy=(phasor_theory.real, phasor_theory.imag),
                 xytext=(phasor_measured.real, phasor_measured.imag),
                 arrowprops=dict(arrowstyle='->', color='tab:grey', linewidth=1))
axes[1].plot(phasor_measured.real, phasor_measured.imag, marker='o', markersize=7,
             linestyle='None', color='tab:red', label='donor, uncalibrated')
axes[1].plot(phasor_theory.real, phasor_theory.imag, marker='*', markersize=13,
             linestyle='None', color='tab:green',
             label=f'donor, calibrated to {donor_lifetime_ns:g} ns')
axes[1].legend(fontsize=8, loc='upper right')
axes[1].set_title(f"{anchor_date} -- calibration anchor")

fig.tight_layout()
save_figure(fig, f"stage2_donor_calibration_{anchor_date}", figure_dir)
plt.show()

## Combined factor per session

The donor fixes the absolute scale in its own session. Every other session
is tied to it by the ratio of the two correction factors:

$$c_\text{session} = c^\text{cal}\;
                     \frac{c^\text{corr}_\text{session}}{c^\text{corr}_\text{anchor}}$$

In [ ]:
total_factor = {
    date: (calibration_factor if date == anchor_date
           else calibration_factor * correction_factor[date] / correction_factor[anchor_date])
    for date in sorted(set(sample_date.values()))
}

pd.DataFrame([{
    'date':         date,
    'correction':   correction_factor[date],
    'total':        factor,
    'rotation_deg': np.angle(factor, deg=True),
    'modulation':   abs(factor),
} for date, factor in total_factor.items()])

## Calibrated phasor table

One row per mask file, with raw and calibrated coordinates and the apparent
lifetimes from the phase ($\tau_\varphi$) and from the modulation ($\tau_m$).

In [ ]:
phasor_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'decay'} for r in records])

raw    = np.asarray(phasor_df['phasor_raw'], dtype=complex)
factor = np.asarray(phasor_df['date'].map(total_factor), dtype=complex)
phasor = raw * factor

phasor_df['g_raw'], phasor_df['s_raw'] = raw.real, raw.imag
phasor_df['g'], phasor_df['s'] = phasor.real, phasor.imag

omega = 2 * np.pi * phasor_df['sample'].map(lambda s: sample_params[s]['sync_rate'])
with np.errstate(divide='ignore', invalid='ignore'):
    modulation_sq = phasor_df['g'] ** 2 + phasor_df['s'] ** 2
    phasor_df['tau_phi_ns'] = (phasor_df['s'] / phasor_df['g']) / omega * 1e9
    phasor_df['tau_m_ns'] = np.sqrt((1 / modulation_sq - 1).clip(lower=0)) / omega * 1e9

phasor_df = phasor_df.drop(columns=['phasor_raw', 'phasor_sum'])

phasor_csv = os.path.join(output_dir, "phasors.csv")
with_display_names(phasor_df).to_csv(phasor_csv, index=False)
print(f"Saved {len(phasor_df)} rows: {phasor_csv}")

with_display_names(phasor_df).head()

## Phasor plot

One point per mask file. The star marks the calibrated donor-only phasor.

In [ ]:
sample_names = sorted(phasor_df['sample'].unique(), key=display_order)
cmap   = plt.get_cmap('tab10' if len(sample_names) <= 10 else 'tab20')
colors = {sample: cmap(i % cmap.N) for i, sample in enumerate(sample_names)}

donor_anchor = calculate_phasor_from_lifetime(donor_lifetime_ns * 1e-9, reference_sync_rate)


def plot_phasors(xlim, ylim, title=None, markersize=4, name=None):
    """IB and cytosol panels over the requested window of the phasor plane."""
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5), layout='constrained')
    entries = {}   # one legend entry per sample, shared by both panels

    for ax, mask_type in zip(axes, ('IB', 'cytosol')):
        draw_universal_circle(ax, sync_rate=reference_sync_rate, tau_max=5,
                              xlim=xlim, ylim=ylim,
                              tick_length=0.02 * (xlim[1] - xlim[0]))
        ax.set_aspect('equal', adjustable='box')
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_title(mask_type)

        for sample in sample_names:
            subset = phasor_df[(phasor_df['mask_type'] == mask_type)
                               & (phasor_df['sample'] == sample)]
            if subset.empty:
                continue
            handle, = ax.plot(subset['g'], subset['s'], linestyle='None', marker='o',
                              markersize=markersize, alpha=0.5,
                              color=colors[sample], label=display_name(sample))
            entries.setdefault(display_name(sample), handle)

        donor_label = f'donor only ({donor_lifetime_ns:g} ns)'
        ax.plot(donor_anchor.real, donor_anchor.imag, linestyle='None',
                marker='*', markersize=13, color='black')
        entries.setdefault(donor_label, plt.Line2D([], [], linestyle='None', marker='*',
                                                   markersize=8, color='black'))

    # Legend beside the panels, in as many columns as keep it within their height.
    legend_fontsize = 8
    rows  = max(1, int(fig.get_figheight() * 72 / (1.6 * legend_fontsize)))
    ncols = max(1, -(-len(entries) // rows))
    fig.legend(list(entries.values()), list(entries), loc='outside right upper',
               fontsize=legend_fontsize, ncols=ncols, frameon=False,
               handletextpad=0.3, columnspacing=1.0, borderaxespad=0.0)

    if title:
        fig.suptitle(title)
    if name:
        save_figure(fig, name, figure_dir)
    plt.show()


plot_phasors(xlim=(0, 1), ylim=(0, 0.8), title="Full phasor plot", name="phasors_full")

In [ ]:
def padded_limits(values, pad=zoom_padding, minimum_span=0.05):
    """(low, high) covering `values` with a relative margin."""
    low, high = float(np.min(values)), float(np.max(values))
    span = max(high - low, minimum_span)
    return low - pad * span, high + pad * span


finite = phasor_df[np.isfinite(phasor_df['g']) & np.isfinite(phasor_df['s'])]

plot_phasors(xlim=padded_limits(finite['g']), ylim=padded_limits(finite['s']),
             title="Phasor plot, cropped to the data", markersize=5, name="phasors_cropped")

## Accumulated decays

All masks of a sample summed into one decay per mask type, exported in long
format (`date, sample, mask_type, time_ns, counts`).

In [ ]:
decay_tables = []
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)

for ax, mask_type in zip(axes, ('IB', 'cytosol')):
    for sample in sample_names:
        decay = aggregate_decay(records, sample, mask_type)
        if decay is None or decay.sum() == 0:
            continue
        params    = sample_params[sample]
        time_axis = (np.arange(params['n_bins']) * params['tcspc_resolution']
                     * params['tcspc_binning'] * 1e9)

        ax.semilogy(time_axis, decay, color=colors[sample],
                    label=f"{display_name(sample)} ({decay.sum():,})")
        decay_tables.append(pd.DataFrame({
            'date':      sample_date[sample],
            'sample':    sample,
            'mask_type': mask_type,
            'time_ns':   time_axis,
            'counts':    decay,
        }))

    ax.set_title(mask_type)
    ax.set_xlabel("time (ns)")
    ax.legend(fontsize=8)

axes[0].set_ylabel("counts")
fig.tight_layout()
save_figure(fig, "decays_by_sample", figure_dir)
plt.show()

decay_df  = pd.concat(decay_tables, ignore_index=True)
decay_csv = os.path.join(output_dir, "decays.csv")
with_display_names(decay_df).to_csv(decay_csv, index=False)
print(f"Saved {len(decay_tables)} curve(s) of {decay_df['sample'].nunique()} sample(s): {decay_csv}")

## Lifetimes and FRET efficiency

From each calibrated phasor $g + i\,s$:

| quantity | definition |
|---|---|
| $\tau_\varphi$ | $s / (\omega g)$ |
| $\tau_m$ | $\sqrt{1/(g^2+s^2) - 1}\;/\;\omega$ |
| $E_\varphi$, $E_m$ | $1 - \tau/\tau_D$ |

Three levels are exported. **Objects** are single labelled IBs or cytosolic
regions. **Images** and **constructs** are pooled from the objects that pass
`min_photons_per_object`, by summing phasor sums and photon counts.

A pooled row holds the photon-weighted value (`tau_phi_ns`, `E_phi`, ...) and
the mean and SD over the level below (`*_mean`, `*_std` over objects;
`*_img` over images). Statistics should use objects or images as replicates.

In [ ]:
FRET_COLUMNS   = ['tau_phi_ns', 'E_phi', 'tau_m_ns', 'E_m', 'outside_circle']
SPREAD_COLUMNS = ['tau_phi_ns', 'E_phi']   # carried up as mean/SD of the level below


def add_fret_columns(df):
    """Apparent lifetimes and FRET efficiencies of the `phasor` column."""
    if df.empty:
        return df.assign(**{c: pd.Series(dtype=float) for c in FRET_COLUMNS})

    tau_donor = donor_lifetime_ns * 1e-9
    pieces = []
    for sample, group in df.groupby('sample', sort=False):
        z = group['phasor'].to_numpy(dtype=complex)
        tau_phi, tau_m = apparent_lifetimes(z, sample_params[sample]['sync_rate'])
        pieces.append(pd.DataFrame({
            'tau_phi_ns':     tau_phi * 1e9,
            'E_phi':          1.0 - tau_phi / tau_donor,
            'tau_m_ns':       tau_m * 1e9,
            'E_m':            1.0 - tau_m / tau_donor,
            'outside_circle': np.abs(z - 0.5) > 0.5 * (1 + 1e-9),
        }, index=group.index))

    return df.join(pd.concat(pieces))


def pool_to_level(source, keys):
    """Photon-pool `source` rows into groups and quantify the pooled phasor."""
    rows = []
    for key, group in source.groupby(keys, sort=True):
        photons = group['photons'].sum()
        rows.append({
            **dict(zip(keys, key)),
            'n_objects':  len(group),
            'photons':    int(photons),
            'phasor':     group['phasor_sum'].sum() / photons if photons else np.nan,
        })
    return add_fret_columns(pd.DataFrame(rows))


def summarize(source, keys, suffix=''):
    """Mean and SD of SPREAD_COLUMNS over `source`, per group."""
    summary = source.groupby(keys, sort=True)[SPREAD_COLUMNS].agg(['mean', 'std'])
    summary.columns = [f'{col}_{stat}{suffix}' for col, stat in summary.columns]
    return summary


IMAGE_KEYS     = ['date', 'sample', 'mask_type', 'ptu_file']
CONSTRUCT_KEYS = ['date', 'sample', 'mask_type']

# ------------------------------------------------------------- level 1: objects
object_df = pd.DataFrame(objects)
factor = np.asarray(object_df['date'].map(total_factor), dtype=complex)
object_df['phasor']     = np.asarray(object_df['phasor_raw'], dtype=complex) * factor
object_df['phasor_sum'] = np.asarray(object_df['phasor_sum'], dtype=complex) * factor
object_df['passed']     = object_df['photons'] >= min_photons_per_object
object_df = add_fret_columns(object_df)

passed = object_df[object_df['passed']].copy()
print(f"Objects: {len(object_df)} total, {len(passed)} with at least "
      f"{min_photons_per_object} photons.")

# --------------------------------------------------------------- level 2: images
image_df = pool_to_level(passed, IMAGE_KEYS)
image_df = image_df.join(summarize(passed, IMAGE_KEYS), on=IMAGE_KEYS)

# ----------------------------------------------------------- level 3: constructs
construct_df = pool_to_level(passed, CONSTRUCT_KEYS)
construct_df = construct_df.join(summarize(passed, CONSTRUCT_KEYS), on=CONSTRUCT_KEYS)
construct_df = construct_df.join(summarize(image_df, CONSTRUCT_KEYS, suffix='_img'),
                                 on=CONSTRUCT_KEYS)
construct_df = construct_df.join(
    image_df.groupby(CONSTRUCT_KEYS).size().rename('n_images'), on=CONSTRUCT_KEYS)


def for_export(frame):
    """Display names, g/s instead of the complex phasor, and the sync rate."""
    phasor = np.asarray(frame['phasor'], dtype=complex)
    frame = frame.assign(
        g=phasor.real,
        s=phasor.imag,
        sync_rate=frame['sample'].map(lambda s: sample_params[s]['sync_rate']),
    )
    return with_display_names(
        frame.drop(columns=[c for c in ('phasor_sum', 'phasor_raw', 'phasor')
                            if c in frame.columns]))


for frame, level in ((object_df, 'objects'), (image_df, 'images'), (construct_df, 'constructs')):
    path = os.path.join(output_dir, f"fret_{level}.csv")
    for_export(frame).to_csv(path, index=False)
    print(f"Saved {len(frame)} rows: {path}")

for_export(construct_df)[['sample', 'mask_type', 'n_images', 'n_objects', 'photons',
                          'tau_phi_ns', 'E_phi', 'tau_m_ns', 'E_m']].round(4)